# (Título interessante) a evolução da representação feminina na política brasileira (2002–2024)

https://github.com/annabrinkhus/projeto_aedv

## 1. Introdução

problema, contexto, motivação e síntese dos achados. Explicar que o sistema é proporcional para deputados e vereadores e majoritário para os demais cargos, e que a cota se aplica às listas partidárias. #provisório

## 2. Dados

### 2.1 Fontes e coleta

Os dados de candidaturas, resultados e prestação de contas vêm do Tribunal Superior Eleitoral (TSE),
acessados por meio da Base dos Dados (dataset `br_tse_eleicoes`), que os disponibiliza já padronizados
e com códigos de município do IBGE harmonizados entre fontes. A coleta foi feita por consultas SQL
ao BigQuery executadas a partir do Python (pacote `basedosdados`). As consultas
estão versionadas na pasta `queries/` do [repositório](https://github.com/annabrinkhus/projeto_aedv), e o notebook
`download.ipynb` reproduz toda a coleta.

### 2.2 Recorte

Consideramos as eleições gerais de 2002 a 2022 e as municipais de 2004 a 2024. A base de candidaturas
tem [N] registros e [K] variáveis, entre elas gênero, cargo, partido, UF, município, idade,
instrução e raça/cor. Os dados de financiamento são consistentes apenas a partir de 2014, e a
raça/cor autodeclarada só consta a partir desse mesmo ano. As candidaturas de 2026 não entram na análise.

### 2.3 Limitações

[Dados anteriores a ~2002 são menos completos; raça/cor só desde 2014; financiamento consistente
só desde 2014; ...]

fontes, anos, tabelas, número de linhas, variáveis usadas, limitações

### 2.4 Leitura dos dados

In [ ]:
import pandas as pd
df = pd.read_parquet("../data/candidatos_completo.parquet")
df.info()

## Limpeza dos dados

In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3375740 entries, 0 to 3375739
Data columns (total 23 columns):
 #   Column                 Dtype 
---  ------                 ----- 
 0   ano                    Int64 
 1   id_eleicao             object
 2   tipo_eleicao           object
 3   data_eleicao           dbdate
 4   sigla_uf               object
 5   sigla_uf_nome          object
 6   id_municipio           object
 7   id_municipio_nome      object
 8   id_municipio_tse       object
 9   id_municipio_tse_nome  object
 10  numero                 object
 11  nome                   object
 12  nome_urna              object
 13  numero_partido         object
 14  sigla_partido          object
 15  cargo                  object
 16  situacao               object
 17  idade                  Int64 
 18  genero                 object
 19  instrucao              object
 20  ocupacao               object
 21  sigla_uf_nascimento    object
 22  raca                   object
dtypes: Int6

In [10]:
for c in ["genero", "cargo", "situacao", "raca", "tipo_eleicao"]:
    print(df[c].value_counts(dropna=False), "\n")

genero
masculino    2421097
feminino      942102
None           12541
Name: count, dtype: int64 

cargo
vereador               2956343
vice-prefeito           118005
prefeito                116016
deputado estadual       114686
deputado federal         54409
deputado distrital        5865
senador                   2201
governador                1611
vice-governador           1609
2º suplente senador       1040
1º suplente senador       1016
2º suplente                901
1º suplente                881
2 suplente                 350
1 suplente                 349
suplente de senador        203
presidente                  99
vice-presidente             93
vice governador             63
Name: count, dtype: int64 

situacao
deferido                                         2730696
None                                              463795
indeferido                                         89777
renuncia                                           59408
indeferido com recurso                    

In [13]:
cols_cat = [
    "tipo_eleicao", "sigla_uf", "sigla_uf_nome", "cargo", "situacao",
    "genero", "instrucao", "ocupacao", "raca", "sigla_partido",
    "sigla_uf_nascimento",
]
for c in cols_cat:
    df[c] = df[c].astype("category")

df["data_eleicao"] = pd.to_datetime(df["data_eleicao"])

In [14]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3375740 entries, 0 to 3375739
Data columns (total 23 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   ano                    Int64         
 1   id_eleicao             object        
 2   tipo_eleicao           category      
 3   data_eleicao           datetime64[ns]
 4   sigla_uf               category      
 5   sigla_uf_nome          category      
 6   id_municipio           object        
 7   id_municipio_nome      object        
 8   id_municipio_tse       object        
 9   id_municipio_tse_nome  object        
 10  numero                 object        
 11  nome                   object        
 12  nome_urna              object        
 13  numero_partido         object        
 14  sigla_partido          category      
 15  cargo                  category      
 16  situacao               category      
 17  idade                  Int64         
 18  genero                

In [15]:
df.head()

,ano,id_eleicao,tipo_eleicao,data_eleicao,sigla_uf,sigla_uf_nome,id_municipio,id_municipio_nome,id_municipio_tse,id_municipio_tse_nome,...,numero_partido,sigla_partido,cargo,situacao,idade,genero,instrucao,ocupacao,sigla_uf_nascimento,raca
0,1994,1994001,eleicao ordinaria,1994-10-03,MS,Mato Grosso do Sul,None,None,None,None,...,15,PMDB,deputado estadual,NaN,29,NaN,NaN,NaN,NaN,NaN
1,1994,1994001,eleicao ordinaria,1994-10-03,BA,Bahia,None,None,None,None,...,25,PFL,deputado estadual,NaN,56,NaN,NaN,NaN,NaN,NaN
2,1994,1994001,eleicao ordinaria,1994-10-03,BA,Bahia,None,None,None,None,...,39,PP,deputado estadual,NaN,49,NaN,NaN,NaN,NaN,NaN
3,1994,1994001,eleicao ordinaria,1994-10-03,NaN,NaN,None,None,None,None,...,11,PPR,vice-presidente,NaN,55,feminino,NaN,NaN,NaN,NaN
4,1994,1994001,eleicao ordinaria,1994-10-03,NaN,NaN,None,None,None,None,...,15,PMDB,vice-presidente,NaN,51,feminino,NaN,NaN,NaN,NaN


In [16]:
df.to_parquet("candidatos_completo.parquet")